In [13]:
#Imports
import torch
import csv
from matplotlib import pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import random

#Data loading
with open("data/mnist_train.csv", "r") as read_obj:
    csv_reader = csv.reader(read_obj)
    train = list(csv_reader)
train = train[1:]
train = [[int(train[i][j]) for j in range(len(train[0]))] for i in range(len(train))]
train = torch.tensor(train, dtype = torch.float32)

#Data formatting and data loader
train_Y = train[:,0].long()
train_X = train[:,1:]

dataset = TensorDataset(train_X, train_Y)

batch_size = 32
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = True)

In [14]:
#Model declarations
#Encoder model
class Encoder(nn.Module):
    def __init__(self, latent_dim = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )

    def forward(self, x):
        return self.net(x)
    
#Predictor model
#TODO add extra variables for additional context for predictor model
class Predictor(nn.Module):
    def __init__(self, latent_dim = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)
        )
    
    def forward(self, x):
        return self.net(x)

#JEPA wrapper module
class MNIST_JEPA(nn.Module):
    def __init__(self, latent_dim = 128, ema = 0.99):
        super().__init__()
        self.context_encoder = Encoder(latent_dim)
        self.target_encoder = Encoder(latent_dim)
        self.predictor = Predictor(latent_dim)
        self.ema = ema

        #update target encoders weights to be identical to context encoder
        self.update_target_encoder(momentum = 0)

        #target encoder will inherit context encoder weights
        #so no need to track gradients
        for param in self.target_encoder.parameters():
            param.requires_grad = False
        
    #updates target encoder weights
    @torch.no_grad()
    def update_target_encoder(self, momentum = None):
        m = self.ema if momentum is None else momentum
        for param_context, param_target in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
            param_target.data = m*param_target + (1-m)*param_context
    
    #TODO make a good masking function for already flattened vectors
    def apply_mask(self, x, block_size = 112):
        start = random.randint(0, 28*28 - block_size)
        x[start: start + block_size] = 0
        return x, start

    def forward(self, x):
        x_masked, action = self.apply_mask(x)
        x_target = x

        z_context = self.context_encoder(x_masked)

        with torch.no_grad():
            z_target = self.target_encoder(x_target)
        
        z_pred = self.predictor(z_context)

        return z_pred, z_target.detach()



In [16]:
#Training setup
device = torch.device("cpu")

model = MNIST_JEPA().to(device)
lr = 1e-3
optimizer = optim.Adam(list(model.context_encoder.parameters()) + list(model.predictor.parameters()), lr = lr)
loss_fn = nn.MSELoss()

#Training
print("Beginning model training . . .")
model.train()
for epoch in range(25):
    total_loss = 0
    for data_X, data_Y in dataloader:

        optimizer.zero_grad()
        
        #forward pass
        z_pred, z_target = model(data_X)

        #loss
        loss = loss_fn(z_pred, z_target)

        loss.backward()
        optimizer.step()

        model.update_target_encoder()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Average Loss: {total_loss/len(dataloader):.4f}")


Beginning model training . . .
Epoch 1 | Average Loss: 224.6622
Epoch 2 | Average Loss: 532.4976
Epoch 3 | Average Loss: 515.7238
Epoch 4 | Average Loss: 490.5699
Epoch 5 | Average Loss: 374.9366
Epoch 6 | Average Loss: 383.3145
Epoch 7 | Average Loss: 290.8404
Epoch 8 | Average Loss: 301.1875
Epoch 9 | Average Loss: 224.8343
Epoch 10 | Average Loss: 238.3152
Epoch 11 | Average Loss: 219.7290
Epoch 12 | Average Loss: 199.4794
Epoch 13 | Average Loss: 187.8401
Epoch 14 | Average Loss: 195.7474
Epoch 15 | Average Loss: 180.2283
Epoch 16 | Average Loss: 182.4107
Epoch 17 | Average Loss: 156.0685
Epoch 18 | Average Loss: 166.6894
Epoch 19 | Average Loss: 162.1242
Epoch 20 | Average Loss: 154.3937
Epoch 21 | Average Loss: 154.5882
Epoch 22 | Average Loss: 150.7846
Epoch 23 | Average Loss: 143.2687
Epoch 24 | Average Loss: 141.1752
Epoch 25 | Average Loss: 143.3736


In [17]:
#Append additional layer to see if the model can be used for digit recognition
class VisionModel(nn.Module):
    def __init__(self, encoder_model, latent_dim = 128, target_dim = 10):
        super().__init__()
        for param in encoder_model.parameters():
            param.requires_grad = False
        
        self.encoder_model = encoder_model

        self.lin = nn.Linear(latent_dim, target_dim)

    def forward(self, x):
        x = self.encoder_model(x)
        x = self.lin(x)
        return x  

In [18]:
#Declaration and training of vision model for 
#downstream labeling
vision_model = VisionModel(model.target_encoder)
vision_optimizer = optim.Adam(vision_model.lin.parameters(), lr = lr)
vision_loss_fn = nn.CrossEntropyLoss()


vision_model.train()
for epoch in range(10):
    total_loss = 0
    running_acc = 0.0
    for data_X, data_Y in dataloader:
        vision_optimizer.zero_grad()

        pred = vision_model(data_X)

        loss = vision_loss_fn(pred, data_Y)

        with torch.no_grad():
            preds = torch.argmax(pred, dim = 1)
            correct = (preds == data_Y)
            accuracy = correct.float().sum()
            running_acc += accuracy

        loss.backward()
        vision_optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Average Loss: {total_loss/len(dataloader):.4f}, Acc: {running_acc/60000:.4f}")

Epoch 1 | Average Loss: 9.5872, Acc: 0.8044
Epoch 2 | Average Loss: 4.1039, Acc: 0.8378
Epoch 3 | Average Loss: 4.1112, Acc: 0.8404
Epoch 4 | Average Loss: 4.0723, Acc: 0.8422
Epoch 5 | Average Loss: 3.9791, Acc: 0.8432
Epoch 6 | Average Loss: 3.9914, Acc: 0.8444
Epoch 7 | Average Loss: 4.0180, Acc: 0.8436
Epoch 8 | Average Loss: 4.0537, Acc: 0.8449
Epoch 9 | Average Loss: 4.0152, Acc: 0.8444
Epoch 10 | Average Loss: 3.9316, Acc: 0.8475
